In [60]:
import pymupdf
from collections import defaultdict

doc = pymupdf.open("/media/hassan/Hassan/Whitepaper/bitcoin.pdf")

# police -> liste de toutes les tailles rencontrées
font_sizes = defaultdict(list)

for page in doc:
    blocks = page.get_text("dict")["blocks"]
    text_blocks = [b for b in blocks if b["type"] == 0]
    
    for block in text_blocks:
        for line in block["lines"]:  # toutes les lignes
            for span in line["spans"]:  # tous les spans
                font_sizes[span["font"]].append({
                    "size": round(span["size"], 1),
                    "text": span["text"][:50],  # aperçu du texte
                    "flags": span["flags"]
                })

# Résumé : police -> tailles distinctes et leur fréquence
print("=== POLICES ET TAILLES ===\n")
for font_name, occurrences in sorted(font_sizes.items()):
    tailles = defaultdict(int)
    for o in occurrences:
        tailles[o["size"]] += 1
    print(f"{font_name}")
    for taille, count in sorted(tailles.items(), reverse=True):
        # Exemple de texte pour cette taille
        exemple = next(o["text"] for o in occurrences if o["size"] == taille)
        print(f"  {taille}pt  x{count:>4}  →  '{exemple}'")
    print()

=== POLICES ET TAILLES ===

ArialMT
  7.2pt  x  12  →  'Block'
  7.1pt  x  29  →  'Block'
  7.0pt  x  28  →  'Transaction'
  6.9pt  x  33  →  'Block'
  6.1pt  x  11  →  'Identities'

CenturySchoolbook-Bold
  14.0pt  x   1  →  'Bitcoin: A Peer-to-Peer Electronic Cash System'
  11.5pt  x  28  →  '1.'

CourierNewPSMT
  8.0pt  x  49  →  '#include <math.h>'

OpenSymbol
  21.3pt  x   4  →  '{'
  16.0pt  x   2  →  '∑'
  11.2pt  x   2  →  ''
  11.0pt  x  23  →  '='
  6.6pt  x  11  →  '='

TimesNewRomanPS-BoldMT
  9.3pt  x   1  →  'Abstract.'

TimesNewRomanPS-ItalicMT
  11.0pt  x  31  →  'q'
  10.1pt  x   5  →  'p'
  9.0pt  x   7  →  '20th Symposium on Information Theory in the Benelu'
  6.6pt  x  11  →  ' z'
  5.9pt  x   1  →  'z'

TimesNewRomanPSMT
  11.0pt  x   4  →  '1'
  10.1pt  x 214  →  'Satoshi Nakamoto'
  9.3pt  x  14  →  '  A purely peer-to-peer version of electronic cash'
  9.0pt  x  25  →  '[1]'
  6.6pt  x   2  →  '0'



In [61]:
from statistics import mode

toutes_tailles = [
    round(span["size"], 1)
    for page in doc
    for block in page.get_text("dict")["blocks"]
    if block["type"] == 0
    for line in block["lines"]
    for span in line["spans"]
]

taille_corps = mode(toutes_tailles)
print(f"Taille corps détectée : {taille_corps}pt")

Taille corps détectée : 10.1pt


In [62]:
import pymupdf
from collections import defaultdict

doc = pymupdf.open("/media/hassan/Hassan/Whitepaper/bitcoin.pdf")

# {font: {size: {"count": int, "flags": {flag_value: count}}}}
font_data = defaultdict(lambda: defaultdict(lambda: {"count": 0, "flags": defaultdict(int)}))

for page in doc:
    blocks = page.get_text("dict")["blocks"]
    text_blocks = [b for b in blocks if b["type"] == 0]
    
    for block in text_blocks:
        for line in block["lines"]:
            for span in line["spans"]:
                font = span["font"]
                size = round(span["size"], 1)
                flags = span["flags"]
                
                font_data[font][size]["count"] += 1
                font_data[font][size]["flags"][flags] += 1

# Affichage lisible
for font, sizes in sorted(font_data.items()):
    print(f"\n{font}")
    for size, data in sorted(sizes.items(), reverse=True):
        flags_str = ", ".join(f"flags={f}(x{c})" for f, c in data["flags"].items())
        print(f"  {size}pt  x{data['count']:>4}  →  {flags_str}")


ArialMT
  7.2pt  x  12  →  flags=4(x12)
  7.1pt  x  29  →  flags=4(x29)
  7.0pt  x  28  →  flags=4(x28)
  6.9pt  x  33  →  flags=4(x33)
  6.1pt  x  11  →  flags=4(x11)

CenturySchoolbook-Bold
  14.0pt  x   1  →  flags=20(x1)
  11.5pt  x  28  →  flags=20(x28)

CourierNewPSMT
  8.0pt  x  49  →  flags=4(x49)

OpenSymbol
  21.3pt  x   4  →  flags=4(x1), flags=5(x3)
  16.0pt  x   2  →  flags=4(x2)
  11.2pt  x   2  →  flags=4(x2)
  11.0pt  x  23  →  flags=4(x22), flags=5(x1)
  6.6pt  x  11  →  flags=4(x11)

TimesNewRomanPS-BoldMT
  9.3pt  x   1  →  flags=20(x1)

TimesNewRomanPS-ItalicMT
  11.0pt  x  31  →  flags=6(x30), flags=7(x1)
  10.1pt  x   5  →  flags=6(x5)
  9.0pt  x   7  →  flags=6(x7)
  6.6pt  x  11  →  flags=6(x11)
  5.9pt  x   1  →  flags=6(x1)

TimesNewRomanPSMT
  11.0pt  x   4  →  flags=4(x4)
  10.1pt  x 214  →  flags=4(x214)
  9.3pt  x  14  →  flags=4(x14)
  9.0pt  x  25  →  flags=4(x25)
  6.6pt  x   2  →  flags=4(x2)


In [65]:
import pymupdf
from collections import defaultdict

doc = pymupdf.open("/media/hassan/Hassan/Whitepaper/bitcoin.pdf")

EXCLUDE_FONTS = {"CourierNewPSMT", "OpenSymbol"}
BOLD_FLAG = 20
BODY_SIZE_MODAL = 10.1

def reconstruct_lines(doc):
    lines_data = []
    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        text_blocks = [b for b in blocks if b["type"] == 0]
        for block in text_blocks:
            for line in block["lines"]:
                spans = line["spans"]
                if not spans:
                    continue
                full_text = "".join(s["text"] for s in spans).strip()
                if not full_text:
                    continue
                clean_spans = [s for s in spans if s["font"] not in EXCLUDE_FONTS]
                if not clean_spans:
                    continue
                dominant_span = max(clean_spans, key=lambda s: len(s["text"]))
                lines_data.append({
                    "page":    page_num + 1,
                    "text":    full_text,
                    "font":    dominant_span["font"],
                    "size":    round(dominant_span["size"], 1),
                    "flags":   dominant_span["flags"],
                    "is_bold": dominant_span["flags"] == BOLD_FLAG,
                    "n_words": len(full_text.split()),
                    "n_spans": len(spans),
                })
    return lines_data

def get_candidates(lines):
    return [l for l in lines if l["is_bold"] and l["size"] > BODY_SIZE_MODAL]

def merge_title_fragments(candidates):
    merged = []
    skip_next = False
    for i, line in enumerate(candidates):
        if skip_next:
            skip_next = False
            continue
        is_number_fragment = (
            line["n_words"] == 1 and
            line["text"].strip().rstrip(".").isdigit()
        )
        if is_number_fragment and i + 1 < len(candidates):
            next_line = candidates[i + 1]
            if (next_line["page"] == line["page"] and
                next_line["size"] == line["size"] and
                next_line["flags"] == line["flags"]):
                merged.append({
                    **line,
                    "text":    f"{line['text']} {next_line['text']}",
                    "n_words": line["n_words"] + next_line["n_words"],
                })
                skip_next = True
                continue
        merged.append(line)
    return merged

# Pipeline
lines      = reconstruct_lines(doc)
candidates = get_candidates(lines)
plan       = merge_title_fragments(candidates)

print("=== PLAN EXTRAIT ===\n")
for entry in plan:
    print(f"  p.{entry['page']:>2}  {entry['size']:>5.1f}pt  '{entry['text']}'")

=== PLAN EXTRAIT ===

  p. 1   14.0pt  'Bitcoin: A Peer-to-Peer Electronic Cash System'
  p. 1   11.5pt  '1. Introduction'
  p. 2   11.5pt  '2. Transactions'
  p. 2   11.5pt  '3. Timestamp Server'
  p. 3   11.5pt  '4. Proof-of-Work'
  p. 3   11.5pt  '5. Network'
  p. 4   11.5pt  '6. Incentive'
  p. 4   11.5pt  '7. Reclaiming Disk Space'
  p. 5   11.5pt  '8. Simplified Payment Verification'
  p. 5   11.5pt  '9. Combining and Splitting Value'
  p. 6   11.5pt  '10. Privacy'
  p. 6   11.5pt  '11. Calculations'
  p. 8   11.5pt  '12. Conclusion'
  p. 9   11.5pt  'References'


In [66]:
import os

whitepaper_dir = "/media/hassan/Hassan/Whitepaper"

for f in sorted(os.listdir(whitepaper_dir)):
    ext = os.path.splitext(f)[1].lower()
    size = os.path.getsize(os.path.join(whitepaper_dir, f))
    print(f"  {ext:>6}  {size/1024:>8.1f} Ko  {f}")

    .pdf    1941.0 Ko  1677580152588.pdf
    .pdf     380.8 Ko  1905.09274v4.pdf
    .pdf     545.5 Ko  DogechainWP.pdf
    .pdf    3274.1 Ko  SHIBPAPER-ABRIDGED-V.1.pdf
    .pdf     267.4 Ko  StableSwap.pdf
    .pdf     244.5 Ko  TerraLunaWP.pdf
    .pdf    4200.2 Ko  binance-coin-whitepaper.pdf
    .pdf     180.0 Ko  bitcoin.pdf
    .pdf     100.2 Ko  document.pdf
    .pdf     247.1 Ko  filecoin-jul-2014.pdf
    .pdf     673.2 Ko  solana-whitepaper.pdf
    .pdf     195.3 Ko  stellar-consensus-protocol.pdf
    .pdf     617.1 Ko  sui.pdf
    .pdf     650.6 Ko  white_paper_v_2_0.pdf
    .pdf    3603.4 Ko  whitepaper-v2.pdf
    .pdf    1465.7 Ko  whitepaper-v3.pdf


In [67]:
import pymupdf
import os
from collections import defaultdict
from statistics import mode

whitepaper_dir = "/media/hassan/Hassan/Whitepaper"

# On prend un échantillon représentatif : léger, moyen, lourd
SAMPLE = [
    "bitcoin.pdf",           # académique LaTeX léger
    "solana-whitepaper.pdf", # technique moyen
    "binance-coin-whitepaper.pdf",  # design lourd
    "DogechainWP.pdf",       # probablement Canva/design
]

def profile_document(path):
    """Profil rapide d'un document : polices, tailles, flags."""
    doc = pymupdf.open(path)
    font_sizes = defaultdict(lambda: defaultdict(int))

    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            if block["type"] != 0:
                continue
            for line in block["lines"]:
                for span in line["spans"]:
                    font_sizes[span["font"]][round(span["size"], 1)] += 1

    # Taille modale = corps de texte
    all_sizes = [
        size
        for sizes in font_sizes.values()
        for size, count in sizes.items()
        for _ in range(count)
    ]
    body_size = mode(all_sizes) if all_sizes else None

    return {
        "name":      os.path.basename(path),
        "pages":     len(doc),
        "body_size": body_size,
        "fonts":     {
            font: dict(sizes)
            for font, sizes in font_sizes.items()
        }
    }

for filename in SAMPLE:
    path = os.path.join(whitepaper_dir, filename)
    profile = profile_document(path)

    print(f"\n{'='*60}")
    print(f"  {profile['name']}  ({profile['pages']} pages)")
    print(f"  body_size détecté : {profile['body_size']}pt")
    print(f"  Polices :")
    for font, sizes in sorted(profile["fonts"].items()):
        sizes_str = "  ".join(
            f"{s}pt x{c}" for s, c in sorted(sizes.items(), reverse=True)
        )
        print(f"    {font:<40} {sizes_str}")


  bitcoin.pdf  (9 pages)
  body_size détecté : 10.1pt
  Polices :
    ArialMT                                  7.2pt x12  7.1pt x29  7.0pt x28  6.9pt x33  6.1pt x11
    CenturySchoolbook-Bold                   14.0pt x1  11.5pt x28
    CourierNewPSMT                           8.0pt x49
    OpenSymbol                               21.3pt x4  16.0pt x2  11.2pt x2  11.0pt x23  6.6pt x11
    TimesNewRomanPS-BoldMT                   9.3pt x1
    TimesNewRomanPS-ItalicMT                 11.0pt x31  10.1pt x5  9.0pt x7  6.6pt x11  5.9pt x1
    TimesNewRomanPSMT                        11.0pt x4  10.1pt x214  9.3pt x14  9.0pt x25  6.6pt x2

  solana-whitepaper.pdf  (32 pages)
  body_size détecté : 12.0pt
  Polices :
    CMBX10                                   10.9pt x1  10.0pt x3
    CMBX12                                   17.2pt x35  14.3pt x92  12.0pt x107
    CMEX10                                   10.0pt x35
    CMMI12                                   12.0pt x18
    CMMI8              

In [ ]:
import pymupdf
import os
import re
from collections import defaultdict
from statistics import mode

# --- CONFIGURATION ---
WHITEPAPER_DIR = "/media/hassan/Hassan/Whitepaper"
OUTPUT_DIR = os.path.join(WHITEPAPER_DIR, "extracted_tocs")
EXCLUDE_FONTS = {"CourierNewPSMT", "OpenSymbol", "CambriaMath", "ChromeSansMM"}

os.makedirs(OUTPUT_DIR, exist_ok=True)

# --- FONCTIONS CŒUR ---

def get_dynamic_body_size(doc):
    """Calcule la taille de police la plus fréquente, en ignorant les très grandes tailles (titres/en-têtes)."""
    all_sizes = []
    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            if block["type"] == 0:
                for line in block["lines"]:
                    for span in line["spans"]:
                        size = round(span["size"], 1)
                        if size <= 15.0:  # Ignore les titres pour ne pas fausser la moyenne
                            all_sizes.append(size)
    return mode(all_sizes) if all_sizes else 10.0

def reconstruct_lines(doc):
    """Extrait les lignes de texte avec leurs métadonnées."""
    lines_data = []
    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        text_blocks = [b for b in blocks if b["type"] == 0]
        
        for block in text_blocks:
            for line in block["lines"]:
                spans = line["spans"]
                if not spans:
                    continue
                
                full_text = "".join(s["text"] for s in spans).strip()
                if not full_text:
                    continue
                
                clean_spans = [s for s in spans if s["font"] not in EXCLUDE_FONTS]
                if not clean_spans:
                    continue
                
                ref_span = clean_spans[0]
                is_bold = (ref_span["flags"] & 16) != 0  
                
                lines_data.append({
                    "page":     page_num + 1,
                    "text":     full_text,
                    "font":     ref_span["font"],
                    "size":     round(ref_span["size"], 1),
                    "flags":    ref_span["flags"],
                    "is_bold":  is_bold,
                    "n_words":  len(full_text.split()),
                })
    return lines_data

def get_candidates(lines, body_size):
    """Filtre intelligent pour ne garder que les vrais titres potentiels."""
    candidates = []
    # Regex pour identifier un numéro de section pur (ex: "1", "1.", "1.1", " 1 ")
    section_pattern = re.compile(r'^\s*\d+(\.\d+)*\.?\s*$')
    
    for l in lines:
        text = l["text"].strip()
        if not text:
            continue
            
        is_heading_style = l["is_bold"] or l["size"] > body_size + 1.0
        is_short_enough = l["n_words"] <= 12
        is_not_code = not re.search(r'[{}]', text)
        
        is_pure_number = bool(section_pattern.match(text))
        
        if is_pure_number:
            # C'est un chiffre : on ne le garde que s'il a un style de titre (gras ou grand)
            # Cela élimine les numéros de page (qui sont petits et non gras)
            if is_heading_style and is_not_code:
                candidates.append(l)
        else:
            # Ce n'est pas un chiffre : on applique les filtres normaux
            if is_heading_style and is_short_enough and is_not_code:
                candidates.append(l)
                
    return candidates

def merge_title_fragments(candidates):
    """Fusionne les numéros de section ("1" ou "1.") avec leur titre ("Introduction")."""
    merged = []
    skip_next = False
    section_pattern = re.compile(r'^\s*\d+(\.\d+)*\.?\s*$')
    
    for i, line in enumerate(candidates):
        if skip_next:
            skip_next = False
            continue
            
        text_stripped = line["text"].strip()
        is_number_fragment = bool(section_pattern.match(text_stripped))
        
        if is_number_fragment and i + 1 < len(candidates):
            next_line = candidates[i + 1]
            next_text_stripped = next_line["text"].strip()
            
            # Vérifie que la ligne suivante n'est pas AUSSI un numéro (évite "1" + "2")
            next_is_number = bool(section_pattern.match(next_text_stripped))
            
            # Condition de fusion : même page, taille similaire (tolérance de 1.5pt), et pas 2 numéros
            if (next_line["page"] == line["page"] and
                abs(next_line["size"] - line["size"]) <= 1.5 and
                not next_is_number):
                
                merged.append({
                    **line,
                    "text":    f"{text_stripped} {next_text_stripped}",
                    "n_words": line["n_words"] + next_line["n_words"],
                })
                skip_next = True
                continue
                
        merged.append(line)
    return merged

def generate_markdown_toc(plan):
    """Convertit le plan en Markdown hiérarchisé."""
    toc_lines = []
    for entry in plan:
        text = entry['text']
        size = entry['size']
        
        if re.match(r'^\s*\d+\.\d+', text):       # Ex: "1.1 Vision"
            prefix = "### "
        elif re.match(r'^\s*\d+\.?\s', text):     # Ex: "1. Introduction" ou "1 Introduction"
            prefix = "## "
        elif size >= 16.0:                        # Ex: "Advanced Decentralized..."
            prefix = "# "
        else:                                     # Ex: "References", "Glossary"
            prefix = "## "
            
        toc_lines.append(f"{prefix}{text}")
        
    return "\n".join(toc_lines)

# --- TRAITEMENT PAR LOT ---

print(f"🔍 Recherche des PDF dans : {WHITEPAPER_DIR}\n")
pdf_files = [f for f in os.listdir(WHITEPAPER_DIR) if f.lower().endswith(".pdf")]

success_count = 0
error_count =0

for filename in sorted(pdf_files):
    filepath = os.path.join(WHITEPAPER_DIR, filename)
    md_filename = os.path.splitext(filename)[0] + ".md"
    md_filepath = os.path.join(OUTPUT_DIR, md_filename)
    
    print(f"📄 Traitement de : {filename}...", end=" ")
    
    try:
        doc = pymupdf.open(filepath)
        body_size = get_dynamic_body_size(doc)
        
        lines = reconstruct_lines(doc)
        candidates = get_candidates(lines, body_size)
        plan = merge_title_fragments(candidates)
        
        markdown_content = f"# Table des Matières : {filename}\n\n"
        markdown_content += generate_markdown_toc(plan)
        
        with open(md_filepath, "w", encoding="utf-8") as f:
            f.write(markdown_content)
            
        doc.close()
        print(f"✅ Succès ({len(plan)} titres, corps={body_size}pt)")
        success_count += 1
        
    except Exception as e:
        print(f"❌ Échec : {e}")
        error_count += 1

print("\n" + "="*50)
print(f"🏁 TRAITEMENT TERMINÉ")
print(f"   ✅ Réussis : {success_count}")
print(f"   ❌ Échecs  : {error_count}")
print(f"   📁 Résultats dans : {OUTPUT_DIR}")

🔍 Recherche des PDF dans : /media/hassan/Hassan/Whitepaper

📄 Traitement de : 1677580152588.pdf... ✅ Succès (41 titres, corps=10.0pt)
📄 Traitement de : 1905.09274v4.pdf... ✅ Succès (44 titres, corps=10.0pt)
📄 Traitement de : DogechainWP.pdf... ✅ Succès (79 titres, corps=11.0pt)
📄 Traitement de : SHIBPAPER-ABRIDGED-V.1.pdf... ✅ Succès (63 titres, corps=9.1pt)
📄 Traitement de : StableSwap.pdf... ✅ Succès (11 titres, corps=10.0pt)
📄 Traitement de : TerraLunaWP.pdf... ✅ Succès (19 titres, corps=10.9pt)
📄 Traitement de : binance-coin-whitepaper.pdf... ✅ Succès (63 titres, corps=11.0pt)
📄 Traitement de : bitcoin.pdf... ✅ Succès (15 titres, corps=10.1pt)
📄 Traitement de : document.pdf... ✅ Succès (274 titres, corps=9.5pt)
📄 Traitement de : filecoin-jul-2014.pdf... ✅ Succès (17 titres, corps=10.0pt)
📄 Traitement de : solana-whitepaper.pdf... ✅ Succès (67 titres, corps=12.0pt)
📄 Traitement de : stellar-consensus-protocol.pdf... ✅ Succès (71 titres, corps=10.0pt)
📄 Traitement de : sui.pdf... ✅ S

In [75]:
import os
import ollama
import time
import traceback

# --- CONFIGURATION ---
INPUT_DIR = "/media/hassan/Hassan/Whitepaper/extracted_tocs"
OUTPUT_DIR = "/media/hassan/Hassan/Whitepaper/refined_tocs"

# Modèle léger et performant
MODEL_NAME = "gemma4:31b-cloud"  # ou "gemma2:9b"

os.makedirs(OUTPUT_DIR, exist_ok=True)

SYSTEM_PROMPT = """
You are an expert document structuring assistant. 
Your task is to clean, format, and hierarchically structure the following raw Table of Contents (TOC) extracted from a PDF.

STRICT RULES:
1. Fix broken numbering: If you see a standalone number (e.g., "1" or "1.") followed by a title on the next line (e.g., "Introduction"), merge them into "1. Introduction".
2. Remove ALL noise: Delete page numbers, glossary definitions, random text fragments, or code snippets that are not actual headings.
3. Use strict Markdown heading levels:
   - `#` for the Main Document Title only.
   - `##` for main sections (e.g., "1. Introduction", "2. Architecture").
   - `###` for subsections (e.g., "1.1 Vision", "2.1 Core").
4. DO NOT invent, guess, or hallucinate any sections that are not explicitly present in the input text.
5. Output ONLY the cleaned Markdown. Do NOT include any conversational text. Start directly with the `#` heading.
"""

def refine_toc_with_llm(raw_toc_text, filename):
    """Envoie le TOC brut à Ollama et retourne le TOC nettoyé."""
    prompt = f"Raw TOC to clean:\n\n{raw_toc_text}"
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": prompt}
            ],
            options={
                "temperature": 0.1,
                "num_predict": 8192,      # ⬅️ AUGMENTÉ pour les plans très longs
                "keep_alive": "15m"       # ⬅️ Garde le modèle en mémoire pour éviter les timeouts
            }
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"\n   ⚠️ DÉTAIL DE L'ERREUR LLM pour {filename} :")
        print(traceback.format_exc())
        return None

# --- TRAITEMENT PAR LOT ---

print(f"🔍 Lecture des plans bruts dans : {INPUT_DIR}\n")
md_files = [f for f in os.listdir(INPUT_DIR) if f.lower().endswith(".md")]

# Astuce : Si vous voulez ne tester QUE le fichier qui a planté, décommentez la ligne ci-dessous :
# md_files = ["whitepaper-v3.md"] 

success_count = 0
error_count = 0

for filename in sorted(md_files):
    input_path = os.path.join(INPUT_DIR, filename)
    output_filename = f"refined_{filename}"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    # Vérifier si le fichier a déjà été traité avec succès pour éviter de le refaire
    if os.path.exists(output_path) and os.path.getsize(output_path) > 100:
        print(f"⏭️  Déjà traité : {filename}")
        success_count += 1
        continue
        
    print(f"🤖 Raffinement de : {filename}...", end=" ", flush=True)
    
    try:
        with open(input_path, "r", encoding="utf-8") as f:
            raw_text = f.read()
            
        print("(Envoi au LLM, cela peut prendre 10-30 secondes pour les longs documents...)", end=" ", flush=True)
        refined_text = refine_toc_with_llm(raw_text, filename)
        
        if refined_text:
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(refined_text)
            print("✅ Succès")
            success_count += 1
        else:
            print("❌ Échec (réponse vide ou tronquée)")
            error_count += 1
            
        # Petite pause pour laisser le GPU respirer
        time.sleep(1)
        
    except Exception as e:
        print(f"\n❌ Échec critique pour {filename} : {e}")
        error_count += 1

print("\n" + "="*60)
print(f"🏁 RAFFINEMENT LLM TERMINÉ")
print(f"   ✅ Réussis : {success_count}")
print(f"   ❌ Échecs  : {error_count}")
print(f"   📁 Plans définitifs dans : {OUTPUT_DIR}")

🔍 Lecture des plans bruts dans : /media/hassan/Hassan/Whitepaper/extracted_tocs

⏭️  Déjà traité : 1677580152588.md
⏭️  Déjà traité : 1905.09274v4.md
⏭️  Déjà traité : DogechainWP.md
⏭️  Déjà traité : SHIBPAPER-ABRIDGED-V.1.md
⏭️  Déjà traité : StableSwap.md
⏭️  Déjà traité : TerraLunaWP.md
⏭️  Déjà traité : binance-coin-whitepaper.md
⏭️  Déjà traité : bitcoin.md
🤖 Raffinement de : document.md... (Envoi au LLM, cela peut prendre 10-30 secondes pour les longs documents...) ✅ Succès
⏭️  Déjà traité : filecoin-jul-2014.md
⏭️  Déjà traité : solana-whitepaper.md
⏭️  Déjà traité : stellar-consensus-protocol.md
⏭️  Déjà traité : sui.md
⏭️  Déjà traité : white_paper_v_2_0.md
⏭️  Déjà traité : whitepaper-v2.md
⏭️  Déjà traité : whitepaper-v3.md

🏁 RAFFINEMENT LLM TERMINÉ
   ✅ Réussis : 16
   ❌ Échecs  : 0
   📁 Plans définitifs dans : /media/hassan/Hassan/Whitepaper/refined_tocs


In [79]:
import os
import pymupdf
import ollama
import time
import traceback
from collections import defaultdict
from statistics import mode
import re

# --- CONFIGURATION ---
WHITEPAPER_DIR = "/media/hassan/Hassan/Whitepaper"
OUTPUT_DIR = "/media/hassan/Hassan/Whitepaper/refined_tocs"
MODEL_NAME = "gemma4:31b-cloud" # Ou votre modèle cloud préféré

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 1. VOIE RAPIDE : Extraction du sommaire natif PDF
# ==========================================
def get_native_toc(doc):
    """
    Tente d'extraire le sommaire natif (bookmarks) du PDF.
    Retourne une chaîne Markdown si le sommaire est riche, sinon None.
    """
    try:
        toc = doc.get_toc()
        # Si le sommaire existe et contient au moins 5 entrées, on le considère valide
        if toc and len(toc) >= 5:
            md_lines = []
            for level, title, page_num in toc:
                # Nettoyer le titre (enlever les retours à la ligne ou espaces bizarres)
                clean_title = " ".join(title.split())
                # Convertir le niveau en Markdown (#, ##, ###)
                # On limite à 6 niveaux max pour la sécurité Markdown
                prefix = "#" * min(level, 6)
                md_lines.append(f"{prefix} {clean_title}")
            return "\n".join(md_lines)
    except Exception:
        pass # Si ça échoue, on passe à la voie de secours
    return None

# ==========================================
# 2. VOIE DE SECOURS : Heuristique + LLM
# ==========================================
def get_dynamic_body_size(doc):
    all_sizes = []
    for page in doc:
        for block in page.get_text("dict")["blocks"]:
            if block["type"] == 0:
                for line in block["lines"]:
                    for span in line["spans"]:
                        size = round(span["size"], 1)
                        if size <= 15.0:
                            all_sizes.append(size)
    return mode(all_sizes) if all_sizes else 10.0

def reconstruct_lines(doc):
    lines_data = []
    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        text_blocks = [b for b in blocks if b["type"] == 0]
        for block in text_blocks:
            for line in block["lines"]:
                spans = line["spans"]
                if not spans: continue
                full_text = "".join(s["text"] for s in spans).strip()
                if not full_text: continue
                
                # On prend le premier span comme référence
                ref_span = spans[0]
                is_bold = (ref_span["flags"] & 16) != 0
                
                lines_data.append({
                    "page": page_num + 1,
                    "text": full_text,
                    "size": round(ref_span["size"], 1),
                    "is_bold": is_bold,
                    "n_words": len(full_text.split()),
                })
    return lines_data

def get_candidates(lines, body_size):
    candidates = []
    section_pattern = re.compile(r'^\s*\d+(\.\d+)*\.?\s*$')
    for l in lines:
        text = l["text"].strip()
        if not text: continue
        
        is_heading_style = l["is_bold"] or l["size"] > body_size + 1.0
        is_short_enough = l["n_words"] <= 12
        is_not_code = not re.search(r'[{}]', text)
        is_pure_number = bool(section_pattern.match(text))
        
        if is_pure_number:
            if is_heading_style and is_not_code:
                candidates.append(l)
        else:
            if is_heading_style and is_short_enough and is_not_code:
                candidates.append(l)
    return candidates

def merge_title_fragments(candidates):
    merged = []
    skip_next = False
    section_pattern = re.compile(r'^\s*\d+(\.\d+)*\.?\s*$')
    
    for i, line in enumerate(candidates):
        if skip_next:
            skip_next = False
            continue
            
        text_stripped = line["text"].strip()
        is_number_fragment = bool(section_pattern.match(text_stripped))
        
        if is_number_fragment and i + 1 < len(candidates):
            next_line = candidates[i + 1]
            next_text_stripped = next_line["text"].strip()
            next_is_number = bool(section_pattern.match(next_text_stripped))
            
            if (next_line["page"] == line["page"] and
                abs(next_line["size"] - line["size"]) <= 1.5 and
                not next_is_number):
                merged.append({
                    **line,
                    "text": f"{text_stripped} {next_text_stripped}",
                })
                skip_next = True
                continue
        merged.append(line)
    return merged

def refine_with_llm(raw_toc_text, filename):
    prompt = f"Raw TOC to clean:\n\n{raw_toc_text}"
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": "You are an expert document structuring assistant. Clean, format, and hierarchically structure this raw TOC into strict Markdown. Merge broken numbering (e.g., '1' + 'Introduction' -> '1. Introduction'). Remove page numbers and noise. Output ONLY the cleaned Markdown."},
                {"role": "user", "content": prompt}
            ],
            options={"temperature": 0.1, "num_predict": 8192, "keep_alive": "15m"}
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"\n   ⚠️ Erreur LLM : {e}")
        return None

# ==========================================
# 3. TRAITEMENT PAR LOT (HYBRIDE)
# ==========================================
print(f"🔍 Traitement des PDF dans : {WHITEPAPER_DIR}\n")
pdf_files = [f for f in os.listdir(WHITEPAPER_DIR) if f.lower().endswith(".pdf")]

success_count = 0
fallback_count = 0
error_count = 0

for filename in sorted(pdf_files):
    filepath = os.path.join(WHITEPAPER_DIR, filename)
    output_filename = f"refined_{os.path.splitext(filename)[0]}.md"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    # Skip si déjà fait
    if os.path.exists(output_path) and os.path.getsize(output_path) > 100:
        print(f"⏭️  Déjà traité : {filename}")
        success_count += 1
        continue
        
    print(f"📄 {filename}...", end=" ", flush=True)
    
    try:
        doc = pymupdf.open(filepath)
        
        # ÉTAPE 1 : Tentative de voie rapide (Natif)
        native_markdown = get_native_toc(doc)
        
        if native_markdown:
            # Succès immédiat ! On sauvegarde et on passe au suivant.
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(f"# Table des Matières : {filename}\n\n{native_markdown}")
            print("✅ Succès (Via sommaire natif PDF, instantané)")
            success_count += 1
            doc.close()
            continue # On passe au fichier suivant, pas besoin de LLM !
            
        # ÉTAPE 2 : Voie de secours (Le sommaire natif était vide ou inexistant)
        print("(Sommaire natif absent, lancement de l'analyse heuristique + LLM...)", end=" ", flush=True)
        
        body_size = get_dynamic_body_size(doc)
        lines = reconstruct_lines(doc)
        candidates = get_candidates(lines, body_size)
        plan = merge_title_fragments(candidates)
        
        # Conversion basique en texte brut pour le LLM
        raw_toc_for_llm = "\n".join([f"{'#' * (1 if l['size'] > 16 else 2)} {l['text']} (p.{l['page']})" for l in plan])
        
        refined_text = refine_with_llm(raw_toc_for_llm, filename)
        
        if refined_text:
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(refined_text)
            print("✅ Succès (Via Heuristique + LLM)")
            fallback_count += 1
        else:
            print("❌ Échec LLM")
            error_count += 1
            
        doc.close()
        time.sleep(0.5)
        
    except Exception as e:
        print(f"\n❌ Échec critique : {e}")
        error_count += 1

print("\n" + "="*60)
print(f"🏁 TRAITEMENT TERMINÉ")
print(f"   ⚡ Traités instantanément (Natif) : {success_count}")
print(f"   🤖 Traités via Heuristique + LLM  : {fallback_count}")
print(f"   ❌ Échecs                          : {error_count}")
print(f"   📁 Résultats dans : {OUTPUT_DIR}")

🔍 Traitement des PDF dans : /media/hassan/Hassan/Whitepaper

📄 1677580152588.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 1905.09274v4.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 DogechainWP.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 SHIBPAPER-ABRIDGED-V.1.pdf... ✅ Succès (Via sommaire natif PDF, instantané)
📄 StableSwap.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 TerraLunaWP.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 binance-coin-whitepaper.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 bitcoin.pdf... (Sommaire natif absent, lancement de l'analyse heuristique + LLM...) ✅ Succès (Via Heuristiq

In [ ]:
# Dernire itération ?
import os
import pymupdf
import ollama
import time
import traceback
from collections import defaultdict
from statistics import mode
import re

# --- CONFIGURATION ---
WHITEPAPER_DIR = "/media/hassan/Hassan/Whitepaper"
OUTPUT_DIR = "/media/hassan/Hassan/Whitepaper/refined_tocs"
MODEL_NAME = "gemma4:31b-cloud" # Ou votre modèle cloud


SECTION_PATTERN = re.compile(r'^\s*(\d+(\.\d+)*|[IVXLCDM]+|[A-Z])\.?\s*$', re.IGNORECASE)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ==========================================
# 1. VOIE RAPIDE : Extraction du sommaire natif PDF
# ==========================================
def get_native_toc(doc):
    try:
        toc = doc.get_toc()
        if toc and len(toc) >= 5:
            md_lines = []
            for level, title, page_num in toc:
                clean_title = " ".join(title.split())
                prefix = "#" * min(level, 6)
                md_lines.append(f"{prefix} {clean_title}")
            return "\n".join(md_lines)
    except Exception:
        pass
    return None

# ==========================================
# 2. VOIE DE SECOURS : Heuristique + LLM
# ==========================================
def get_dynamic_body_size(doc, sample_pages=10):
    all_sizes = []
    for page in doc[:sample_pages]:
        for block in page.get_text("dict")["blocks"]:
            if block["type"] == 0:
                for line in block["lines"]:
                    for span in line["spans"]:
                        size = round(span["size"], 1)
                        if size <= 15.0:
                            all_sizes.append(size)
    return mode(all_sizes) if all_sizes else 10.0

def reconstruct_lines(doc):
    lines_data = []
    for page_num, page in enumerate(doc):
        blocks = page.get_text("dict")["blocks"]
        text_blocks = [b for b in blocks if b["type"] == 0]
        for block in text_blocks:
            for line in block["lines"]:
                spans = line["spans"]
                if not spans: continue
                full_text = "".join(s["text"] for s in spans).strip()
                if not full_text: continue
                
                ref_span = spans[0]
                is_bold = (ref_span["flags"] & 16) != 0
                
                lines_data.append({
                    "page": page_num + 1,
                    "text": full_text,
                    "size": round(ref_span["size"], 1),
                    "is_bold": is_bold,
                    "n_words": len(full_text.split()),
                })
    return lines_data

def get_candidates(lines, body_size):
    """Filtrage TRÈS strict pour ne garder que des titres potentiels."""
    candidates = []

    for l in lines:
        text = l["text"].strip()
        if not text: continue
        
        # 1. Doit ressembler à un titre (gras ou grand)
        is_heading_style = l["is_bold"] or l["size"] > body_size + 1.0
        
        # 2. Doit être COURT (max 8 mots). Un titre n'est pas une phrase.
        is_short_enough = l["n_words"] <= 8
        
        # 3. Ne doit PAS être une phrase complète (ne finit pas par un point, sauf si c'est "1.")
        is_not_sentence = not (text.endswith('.') and l["n_words"] > 3)
        
        # 4. Ne doit PAS être une liste à puces ou une paire clé-valeur (ex: "CA: 0x...")
        is_not_list_or_kv = not re.match(r'^[\*\-\•]\s|^\w+\s*:', text)
        
        # 5. Ne doit PAS contenir de code
        is_not_code = not re.search(r'[{}]', text)
        
        is_pure_number = bool(SECTION_PATTERN.match(text))
        
        if is_pure_number:
            if is_heading_style and is_not_code:
                candidates.append(l)
        else:
            if is_heading_style and is_short_enough and is_not_sentence and is_not_list_or_kv and is_not_code:
                candidates.append(l)
                
    return candidates

def merge_title_fragments(candidates):
    merged = []
    skip_next = False
    
    for i, line in enumerate(candidates):
        if skip_next:
            skip_next = False
            continue
            
        text_stripped = line["text"].strip()
        is_number_fragment = bool(SECTION_PATTERN.match(text_stripped))
        
        if is_number_fragment and i + 1 < len(candidates):
            next_line = candidates[i + 1]
            next_text_stripped = next_line["text"].strip()
            next_is_number = bool(SECTION_PATTERN.match(next_text_stripped))
            
            if (next_line["page"] == line["page"] and
                abs(next_line["size"] - line["size"]) <= 1.5 and
                not next_is_number):
                merged.append({
                    **line,
                    "text": f"{text_stripped} {next_text_stripped}",
                })
                skip_next = True
                continue
        merged.append(line)
    return merged

def refine_with_llm(raw_toc_text, filename):
    """Prompt blindé avec exemples pour forcer la sortie strictement en titres."""
    
    system_prompt = """You are a strict Table of Contents extractor. 
Your ONLY task is to output a hierarchical Markdown list of HEADINGS.

STRICT RULES:
1. OUTPUT ONLY LINES STARTING WITH '#', '##', or '###'.
2. DO NOT output any body text, paragraphs, descriptions, dates, or bullet points.
3. DO NOT output any metadata like "CA:", "Tax:", "Total Supply:", or contract addresses.
4. Merge broken numbering (e.g., "1" followed by "About" becomes "## 1. About").
5. If the input contains full sentences or lists, IGNORE them completely. Only keep the section titles.

EXAMPLE INPUT:
1
Introduction
This is a paragraph about the intro.
1.1 Background
The background is...
* Tax: 7%
* Supply: 420M

EXAMPLE OUTPUT:
## 1. Introduction
### 1.1 Background
"""
    
    prompt = f"Raw text to extract headings from:\n\n{raw_toc_text}"
    
    try:
        response = ollama.chat(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": prompt}
            ],
            options={
                "temperature": 0.0,       # 0.0 pour un déterminisme absolu
                "num_predict": 2048,      # Suffisant pour un plan, évite les coupures
                "keep_alive": "15m"
            }
        )
        return response['message']['content'].strip()
    except Exception as e:
        print(f"\n   ⚠️ Erreur LLM : {e}")
        return None

# ==========================================
# 3. TRAITEMENT PAR LOT (HYBRIDE)
# ==========================================
print(f"🔍 Traitement des PDF dans : {WHITEPAPER_DIR}\n")
pdf_files = [f for f in os.listdir(WHITEPAPER_DIR) if f.lower().endswith(".pdf")]

success_count = 0
fallback_count = 0
error_count = 0

for filename in sorted(pdf_files):
    filepath = os.path.join(WHITEPAPER_DIR, filename)
    output_filename = f"refined_{os.path.splitext(filename)[0]}.md"
    output_path = os.path.join(OUTPUT_DIR, output_filename)
    
    if os.path.exists(output_path) and os.path.getsize(output_path) > 100:
        print(f"⏭️  Déjà traité : {filename}")
        success_count += 1
        continue
        
    print(f"📄 {filename}...", end=" ", flush=True)
    
    try:
        doc = pymupdf.open(filepath)
        native_markdown = get_native_toc(doc)
        
        if native_markdown:
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(f"# Table des Matières : {filename}\n\n{native_markdown}")
            print("✅ Succès (Via sommaire natif PDF)")
            success_count += 1
            doc.close()
            continue
            
        print("(Analyse heuristique + LLM...)", end=" ", flush=True)
        
        body_size = get_dynamic_body_size(doc)
        lines = reconstruct_lines(doc)
        candidates = get_candidates(lines, body_size)
        plan = merge_title_fragments(candidates)
        
        # On envoie au LLM uniquement les textes candidats, avec leur taille pour l'indice de hiérarchie
        raw_toc_for_llm = "\n".join([f"{'#' if l['size'] > body_size * 1.4 else '##'} {l['text']}" for l in plan])
        
        refined_text = refine_with_llm(raw_toc_for_llm, filename)
        
        if refined_text:
            # Nettoyage final de sécurité : on s'assure que le fichier ne contient que des #
            final_lines = [line for line in refined_text.split('\n') if line.strip().startswith('#') or line.strip() == ""]
            final_text = "\n".join(final_lines)
            
            with open(output_path, "w", encoding="utf-8") as f:
                f.write(f"# Table des Matières : {filename}\n\n{final_text}")
            print("✅ Succès (Via Heuristique + LLM)")
            fallback_count += 1
        else:
            print("❌ Échec LLM")
            error_count += 1
            
        doc.close()
        time.sleep(0.5)
        
    except Exception as e:
        print(f"\n❌ Échec critique : {e}")
        error_count += 1

print("\n" + "="*60)
print(f"🏁 TRAITEMENT TERMINÉ")
print(f"   ⚡ Traités instantanément (Natif) : {success_count}")
print(f"   🤖 Traités via Heuristique + LLM  : {fallback_count}")
print(f"   ❌ Échecs                          : {error_count}")

🔍 Traitement des PDF dans : /media/hassan/Hassan/Whitepaper

📄 1677580152588.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 1905.09274v4.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 DogechainWP.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 SHIBPAPER-ABRIDGED-V.1.pdf... ✅ Succès (Via sommaire natif PDF)
📄 StableSwap.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 TerraLunaWP.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 binance-coin-whitepaper.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 bitcoin.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 document.pdf... ✅ Succès (Via sommaire natif PDF)
📄 filecoin-jul-2014.pdf... (Analyse heuristique + LLM...) ✅ Succès (Via Heuristique + LLM)
📄 solana-whitepaper.pdf... ✅ Succès (Via sommaire natif PDF)
📄 stellar-consensus-protocol.pdf... ✅ Succès (Via sommaire natif

In [4]:
# Claude ne fonctionne pas pout le moment
import pymupdf
import os
import re
from collections import defaultdict
from statistics import mode
from dataclasses import dataclass, field
from typing import Optional
import ollama

# ==========================================
# STRUCTURES DE DONNÉES
# ==========================================

@dataclass
class SpanData:
    """Représente un span brut extrait du PDF."""
    text: str
    font: str
    size: float
    flags: int
    page: int

    @property
    def is_bold(self) -> bool:
        return bool(self.flags & (1 << 4))

    @property
    def is_italic(self) -> bool:
        return bool(self.flags & (1 << 1))

    @property
    def is_superscript(self) -> bool:
        return bool(self.flags & (1 << 0))


@dataclass
class LineCandidate:
    """
    Représente une ligne reconstruite avec ses métadonnées
    et les signaux qui l'ont qualifiée comme candidat titre.
    """
    text: str
    page: int
    size: float
    is_bold: bool
    n_words: int
    signals: list[str] = field(default_factory=list)  # signaux de qualification
    level: Optional[int] = None                        # niveau hiérarchique (1, 2, 3)


@dataclass
class Section:
    """Section finale avec titre et contenu associé."""
    title: str
    level: int
    page: int
    content: str = ""
    children: list["Section"] = field(default_factory=list)


# ==========================================
# EXTRACTEUR PRINCIPAL
# ==========================================

class WhitepaperExtractor:
    """
    Pipeline d'extraction de structure pour whitepapers crypto.
    
    Stratégie :
        1. Sommaire natif PDF (rapide, fiable)
        2. Heuristique visuelle (polices, tailles, flags)
        3. LLM arbitre sur les candidats ambigus (Gemma)
    """

    # Polices parasites à exclure systématiquement
    EXCLUDE_FONTS = {"CourierNewPSMT", "OpenSymbol", "Courier", "CourierNew"}

    # Pattern numérotation : "1.", "1.1", "1.1.1", "A.", "I.", "a)"
    SECTION_NUMBER_PATTERN = re.compile(
        r'^\s*(\d+(\.\d+)*\.?|[A-Z]\.?|[IVXivx]+\.?)\s*$'
    )

    # Exclusions explicites indépendantes du scoring
    EXCLUDE_PATTERNS = [
        re.compile(r'^0x[a-fA-F0-9]{10,}'),     # adresses crypto
        re.compile(r'https?://'),                  # URLs
        re.compile(r'[{}]'),                       # code
        re.compile(r'^\s*[\*\-\•]\s'),            # listes à puces
        re.compile(r'^\w[\w\s]*\s*:\s+0x'),       # "CA: 0x..."
        re.compile(r'^\d+[\.,]\d+\s*%'),          # pourcentages
    ]

    def __init__(
        self,
        llm_model: str = "gemma4:31b-cloud",
        llm_temperature: float = 0.0,
        body_size_cap: float = 15.0,   # taille max pour le calcul du corps
        size_ratio_h1: float = 1.5,    # ratio size/body_size pour H1
        size_ratio_h2: float = 1.15,   # ratio pour H2
        max_title_words: int = 12,
        min_toc_entries: int = 5,      # seuil sommaire natif
    ):
        self.llm_model = llm_model
        self.llm_temperature = llm_temperature
        self.body_size_cap = body_size_cap
        self.size_ratio_h1 = size_ratio_h1
        self.size_ratio_h2 = size_ratio_h2
        self.max_title_words = max_title_words
        self.min_toc_entries = min_toc_entries

    # ------------------------------------------
    # ÉTAPE 0 : Sommaire natif
    # ------------------------------------------

    def _get_native_toc(self, doc: pymupdf.Document) -> Optional[list[Section]]:
        """
        Tente d'extraire le sommaire natif du PDF.
        Retourne None si absent ou insuffisant.
        """
        try:
            toc = doc.get_toc()
            if toc and len(toc) >= self.min_toc_entries:
                sections = []
                for level, title, page_num in toc:
                    clean_title = " ".join(title.split())
                    if clean_title:
                        sections.append(Section(
                            title=clean_title,
                            level=min(level, 3),
                            page=page_num
                        ))
                return sections
        except Exception:
            pass
        return None

    # ------------------------------------------
    # ÉTAPE 1 : Calcul dynamique du corps de texte
    # ------------------------------------------

    def _get_body_size(self, doc: pymupdf.Document) -> float:
        """
        Calcule la taille de police modale = corps de texte.
        Filtre les tailles > body_size_cap pour éviter que
        les grands titres biaisent le calcul.
        """
        all_sizes = []
        for page in doc:
            for block in page.get_text("dict")["blocks"]:
                if block["type"] != 0:
                    continue
                for line in block["lines"]:
                    for span in line["spans"]:
                        size = round(span["size"], 1)
                        if size <= self.body_size_cap:
                            all_sizes.append(size)
        return mode(all_sizes) if all_sizes else 10.0

    # ------------------------------------------
    # ÉTAPE 2 : Reconstruction des lignes
    # ------------------------------------------

    def _reconstruct_lines(self, doc: pymupdf.Document) -> list[dict]:
        """
        Reconstruit les lignes complètes depuis les spans.
        Le span dominant (le plus long) détermine les métadonnées.
        """
        lines_data = []

        for page_num, page in enumerate(doc):
            for block in page.get_text("dict")["blocks"]:
                if block["type"] != 0:
                    continue
                for line in block["lines"]:
                    spans = line["spans"]
                    if not spans:
                        continue

                    full_text = "".join(s["text"] for s in spans).strip()
                    if not full_text:
                        continue

                    # Filtrer les polices parasites
                    clean_spans = [
                        s for s in spans
                        if s["font"] not in self.EXCLUDE_FONTS
                    ]
                    if not clean_spans:
                        continue

                    # Span dominant = le plus représentatif de la ligne
                    dominant = max(clean_spans, key=lambda s: len(s["text"]))

                    lines_data.append({
                        "page":    page_num + 1,
                        "text":    full_text,
                        "font":    dominant["font"],
                        "size":    round(dominant["size"], 1),
                        "flags":   dominant["flags"],
                        "is_bold": bool(dominant["flags"] & (1 << 4)),
                        "n_words": len(full_text.split()),
                    })

        return lines_data

    # ------------------------------------------
    # ÉTAPE 3 : Qualification des candidats
    # ------------------------------------------

    def _is_excluded(self, text: str) -> bool:
        """Vérifie si une ligne doit être exclue sans ambiguïté."""
        return any(p.search(text) for p in self.EXCLUDE_PATTERNS)

    def _get_candidates(
        self,
        lines: list[dict],
        body_size: float
    ) -> list[LineCandidate]:
        """
        Qualifie les candidats titres par accumulation de signaux.
        Une ligne est candidate si elle a AU MOINS UN signal fort,
        ou une combinaison de signaux faibles.
        """
        candidates = []

        for line in lines:
            text = line["text"].strip()
            if not text or self._is_excluded(text):
                continue

            signals = []

            # --- Signaux forts (suffisants seuls) ---

            if line["is_bold"] and line["size"] > body_size:
                signals.append("bold_large")

            if line["size"] >= body_size * self.size_ratio_h1:
                signals.append("size_h1")

            if line["size"] >= body_size * self.size_ratio_h2:
                signals.append("size_h2")

            if bool(self.SECTION_NUMBER_PATTERN.match(text)):
                signals.append("numbered")

            # --- Signaux faibles (nécessitent combinaison) ---

            if line["n_words"] <= 6:
                signals.append("short")

            if not text.endswith(('.', ',', ';', ':')):
                signals.append("no_punctuation")

            if text.isupper() and len(text) > 2:
                signals.append("all_caps")

            if line["is_bold"]:
                signals.append("bold")

            # --- Règle de qualification ---
            # Signal fort seul → candidat
            has_strong = any(s in signals for s in (
                "bold_large", "size_h1", "size_h2"
            ))
            # Combinaison de signaux faibles → candidat
            weak_count = sum(1 for s in signals if s in (
                "short", "no_punctuation", "all_caps", "bold", "numbered"
            ))
            has_weak_combo = weak_count >= 3

            if not (has_strong or has_weak_combo):
                continue

            # Filtre final : longueur max
            if line["n_words"] > self.max_title_words:
                continue

            candidates.append(LineCandidate(
                text=text,
                page=line["page"],
                size=line["size"],
                is_bold=line["is_bold"],
                n_words=line["n_words"],
                signals=signals,
            ))

        return candidates

    # ------------------------------------------
    # ÉTAPE 4 : Fusion des fragments
    # ------------------------------------------

    def _merge_fragments(
        self,
        candidates: list[LineCandidate]
    ) -> list[LineCandidate]:
        """
        Fusionne les fragments de titres séparés :
        '1.' + 'Introduction' → '1. Introduction'
        """
        merged = []
        skip_next = False

        for i, line in enumerate(candidates):
            if skip_next:
                skip_next = False
                continue

            is_fragment = bool(
                self.SECTION_NUMBER_PATTERN.match(line.text.strip())
            )

            if is_fragment and i + 1 < len(candidates):
                next_line = candidates[i + 1]
                next_is_fragment = bool(
                    self.SECTION_NUMBER_PATTERN.match(next_line.text.strip())
                )

                same_page = next_line.page == line.page
                similar_size = abs(next_line.size - line.size) <= 1.5
                next_not_fragment = not next_is_fragment

                if same_page and similar_size and next_not_fragment:
                    merged.append(LineCandidate(
                        text=f"{line.text.strip()} {next_line.text.strip()}",
                        page=line.page,
                        size=line.size,
                        is_bold=line.is_bold,
                        n_words=line.n_words + next_line.n_words,
                        signals=list(set(line.signals + next_line.signals)),
                    ))
                    skip_next = True
                    continue

            merged.append(line)

        return merged

    # ------------------------------------------
    # ÉTAPE 5 : Inférence de la hiérarchie
    # ------------------------------------------

    def _infer_levels(
        self,
        candidates: list[LineCandidate],
        body_size: float
    ) -> list[LineCandidate]:
        """
        Attribue un niveau hiérarchique relatif à body_size.
        Évite les seuils absolus arbitraires.
        """
        for c in candidates:
            ratio = c.size / body_size
            if ratio >= self.size_ratio_h1 or (c.is_bold and ratio >= 1.3):
                c.level = 1
            elif ratio >= self.size_ratio_h2 or c.is_bold:
                c.level = 2
            else:
                c.level = 3
        return candidates

    # ------------------------------------------
    # ÉTAPE 6 : Arbitrage LLM
    # ------------------------------------------

    def _build_llm_prompt(self, candidates: list[LineCandidate]) -> str:
        """Construit le contexte candidats pour le LLM."""
        lines = []
        for c in candidates:
            signals_str = ",".join(c.signals)
            lines.append(
                f"[p{c.page}|L{c.level}|{c.size}pt|{signals_str}] {c.text}"
            )
        return "\n".join(lines)

    def _call_llm(self, candidates: list[LineCandidate]) -> Optional[list[Section]]:
        """
        Envoie les candidats au LLM pour arbitrage de la hiérarchie.
        Retourne une liste de Section ou None si échec.
        """
        prompt_content = self._build_llm_prompt(candidates)

        system_prompt = """You are a document structure analyzer for crypto whitepapers.
You receive a list of title candidates extracted from a PDF, with metadata:
[page|level_hint|size|signals] text

Your task:
1. Confirm which lines are real section titles (remove false positives)
2. Assign the correct hierarchy level (1, 2 or 3)
3. Return ONLY valid JSON, no markdown, no explanation

Output format:
[
  {"title": "1. Introduction", "level": 1, "page": 2},
  {"title": "1.1 Background", "level": 2, "page": 2}
]

Rules:
- Numbered sections like "1.", "1.1" are strong signals
- Short bold lines are likely titles
- Ignore addresses, URLs, percentages, bullet points
- Use relative size to infer hierarchy when numbering is absent"""

        try:
            response = ollama.chat(
                model=self.llm_model,
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user",   "content": prompt_content}
                ],
                options={
                    "temperature": self.llm_temperature,
                    "num_predict": 2048,
                }
            )
            raw = response["message"]["content"].strip()

            # Nettoyage : extraire le JSON même si le modèle bavarde
            json_match = re.search(r'\[.*\]', raw, re.DOTALL)
            if not json_match:
                return None

            import json
            data = json.loads(json_match.group())
            return [
                Section(
                    title=item["title"],
                    level=item.get("level", 2),
                    page=item.get("page", 0)
                )
                for item in data
                if "title" in item
            ]

        except Exception as e:
            print(f"LLM error: {e}")
            return None

    # ------------------------------------------
    # POINT D'ENTRÉE PUBLIC
    # ------------------------------------------

    def extract(
        self,
        pdf_path: str,
        use_llm: bool = True
    ) -> list[Section]:
        """
        Extrait le plan structuré d'un whitepaper PDF.

        Args:
            pdf_path:  chemin vers le PDF
            use_llm:   si False, retourne le résultat heuristique brut

        Returns:
            Liste de Section avec titre, niveau, page
        """
        doc = pymupdf.open(pdf_path)

        # Voie rapide
        native = self._get_native_toc(doc)
        if native:
            doc.close()
            return native

        # Voie heuristique
        body_size  = self._get_body_size(doc)
        lines      = self._reconstruct_lines(doc)
        candidates = self._get_candidates(lines, body_size)
        candidates = self._merge_fragments(candidates)
        candidates = self._infer_levels(candidates, body_size)
        doc.close()

        if not use_llm or not candidates:
            return [
                Section(title=c.text, level=c.level or 2, page=c.page)
                for c in candidates
            ]

        # Arbitrage LLM sur les candidats
        sections = self._call_llm(candidates)

        # Fallback heuristique si LLM échoue
        if not sections:
            return [
                Section(title=c.text, level=c.level or 2, page=c.page)
                for c in candidates
            ]

        return sections


# ==========================================
# USAGE
# ==========================================

if __name__ == "__main__":
    import json

    extractor = WhitepaperExtractor(llm_model="gemma4:31b-cloud")

    WHITEPAPER_DIR = "/media/hassan/Hassan/Whitepaper"
    OUTPUT_DIR     = "/media/hassan/Hassan/Whitepaper/claude_tocs"
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    for filename in sorted(os.listdir(WHITEPAPER_DIR)):
        if not filename.endswith(".pdf"):
            continue

        path = os.path.join(WHITEPAPER_DIR, filename)
        stem = os.path.splitext(filename)[0]

        # Chemins de sortie
        out_md   = os.path.join(OUTPUT_DIR, f"{stem}.md")
        out_json = os.path.join(OUTPUT_DIR, f"{stem}.json")

        # Skip si déjà traité
        if os.path.exists(out_md):
            print(f"⏭  déjà traité : {filename}")
            continue

        print(f"\n{'='*60}\n  {filename}\n{'='*60}")

        sections = extractor.extract(path, use_llm=True)

        # --- Markdown ---
        md_lines = [f"# Table des matières : {filename}\n"]
        for s in sections:
            indent = "  " * (s.level - 1)
            md_lines.append(f"{indent}{'#' * s.level} {s.title}  (p.{s.page})")

        with open(out_md, "w", encoding="utf-8") as f:
            f.write("\n".join(md_lines))

        # --- JSON (utile pour alimenter analyze_document ensuite) ---
        json_data = [
            {"title": s.title, "level": s.level, "page": s.page}
            for s in sections
        ]
        with open(out_json, "w", encoding="utf-8") as f:
            json.dump(json_data, f, ensure_ascii=False, indent=2)

        print(f"  → {out_md}")
        print(f"  → {out_json}")
        for s in sections:
            indent = "  " * (s.level - 1)
            print(f"{indent}{'#' * s.level} {s.title}  (p.{s.page})")


  1677580152588.pdf
  → /media/hassan/Hassan/Whitepaper/claude_tocs/1677580152588.md
  → /media/hassan/Hassan/Whitepaper/claude_tocs/1677580152588.json
# ABOUT  (p.2)
# Why should I own FLOKI CEO Token?  (p.3)
  ## Instant Usage Rewards  (p.3)
  ## Safe and Secure  (p.3)
  ## Renounce Ownership  (p.3)
# Tokenomics  (p.4)
  ## $FLOKICEO  (p.4)
  ## Total supply  (p.4)
  ## Liquidity Unlocked Time  (p.4)
# Roadmap  (p.5)

  1905.09274v4.pdf
  → /media/hassan/Hassan/Whitepaper/claude_tocs/1905.09274v4.md
  → /media/hassan/Hassan/Whitepaper/claude_tocs/1905.09274v4.json
# Abstract  (p.1)
# 1 Introduction  (p.1)
# 2 Background  (p.2)
  ## 2.1 Blockchains  (p.2)
  ## 2.2 Sampling-Based Data Availability  (p.3)
# 3 Model  (p.4)
  ## 3.1 Threat Model and Nodes  (p.4)
  ## 3.2 Block Model  (p.4)
  ## 3.3 Goals  (p.4)
# 4 Block Validity Rule Design  (p.5)
  ## 4.1 Simplistic Validity Rule  (p.5)
  ## 4.2 Probabilistic Validity Rule  (p.6)
# 5 Application-Layer Design  (p.6)
  ## 5.1 Application